# Multi-Temporal Satellite Crop Classification with TSViT-DINOv3
### End-to-End Google Colab Training Pipeline
- **Backbone:** DINO-v3 Spatial Encoder + Factorized Temporal Transformer
- **Input:** Sentinel-2 10-Band Time Series $[T, 10, 24, 24]$ with Day of Year (DOY)
- **Target:** RPG 2022 Agricultural Crop Classification Map $[24, 24]$
- **Exports:** PyTorch Checkpoint (`.pt`) and ONNX Model (`.onnx`)

### 1. Check GPU Environment

In [ ]:
!nvidia-smi
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

### 2. Clone Repository & Install Dependencies

In [ ]:
# Clone repo (or pull latest changes if already present)
import os
REPO_NAME = "crop_classification_tsvit_dinov3"
if not os.path.exists(REPO_NAME):
    !git clone https://github.com/your-username/crop_classification_tsvit_dinov3.git
    %cd {REPO_NAME}
else:
    %cd {REPO_NAME}
    !git pull

# Install core packages
!pip install -q einops timm rasterio shapely pyyaml kaggle onnx scikit-learn matplotlib

### 3. Download Dataset from Kaggle

In [ ]:
import os
from google.colab import files

DATA_DIR = "data/processed"
os.makedirs(DATA_DIR, exist_ok=True)

# Check for Kaggle credentials
KAGGLE_PATH = os.path.expanduser("~/.kaggle/kaggle.json")
if not os.path.exists(KAGGLE_PATH):
    print("Please upload your kaggle.json file:")
    uploaded = files.upload()
    for fn in uploaded.keys():
        os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)
        os.rename(fn, KAGGLE_PATH)
        os.chmod(KAGGLE_PATH, 0o600)

# Download Kaggle dataset (replace with your Kaggle dataset slug)
KAGGLE_DATASET = "username/sentinel2-tsvit-crop-classification-24x24"
try:
    !kaggle datasets download -d {KAGGLE_DATASET} -p {DATA_DIR} --unzip
except Exception as e:
    print("Kaggle download warning:", e)

# Fallback check: if no files present, generate high-fidelity synthetic benchmark dataset
import glob
if len(glob.glob(os.path.join(DATA_DIR, "*.pt"))) == 0:
    print("Generating local synthetic dataset for demonstration and pipeline testing...")
    from src.data.dataset_builder import S2CropDatasetBuilder
    builder = S2CropDatasetBuilder(patch_size=24)
    builder.build_synthetic_dataset(
        num_timestamps=30,
        height=144,
        width=144,
        num_samples_target=36,
        output_dir=DATA_DIR,
        save_format="pt"
    )
    print("Synthetic dataset ready in:", DATA_DIR)

### 4. Setup DataLoaders & Verify Tensor Shapes

In [ ]:
import sys
sys.path.insert(0, ".")

from src.data.dataloader import get_dataloaders
from src.data.constants import CROP_CLASSES_VI, NUM_CLASSES

train_loader, val_loader = get_dataloaders(DATA_DIR, batch_size=8, num_workers=2, val_split=0.2)
print(f"Train batches: {len(train_loader)} | Val batches: {len(val_loader)}")

sample_batch = next(iter(train_loader))
print("Imagery Shape [B, T, 10, H, W]:", sample_batch['inputs'].shape)
print("DOY Vector Shape [B, T]:       ", sample_batch['doy'].shape)
print("Crop Label Shape [B, H, W]:    ", sample_batch['label'].shape)

### 5. Initialize TSViT-DINOv3 Model

In [ ]:
import torch.nn as nn
from src.models.tsvit_dinov3 import TSViT_DINOv3

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TSViT_DINOv3(
    img_res=24,
    patch_size=2,
    in_channels=10,
    num_classes=NUM_CLASSES,
    dim=192,
    temporal_depth=4,
    spatial_depth=4,
    heads=4,
    dim_head=48,
    dropout=0.1
).to(device)

num_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model Initialized on {device}. Total Trainable Parameters: {num_params:,}")

### 6. Training & Validation Loop

In [ ]:
import torch.optim as optim
from src.utils.metrics import compute_segmentation_metrics
from src.utils.export import save_checkpoint, export_onnx

EPOCHS = 15
LR = 1e-3
criterion = nn.CrossEntropyLoss(ignore_index=0)  # Ignore background class 0
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-5)

best_f1 = 0.0
history = {'train_loss': [], 'val_loss': [], 'val_f1': [], 'val_iou': []}

for epoch in range(1, EPOCHS + 1):
    model.train()
    train_loss = 0.0
    for batch in train_loader:
        x = batch['inputs'].to(device)
        doy = batch['doy'].to(device)
        y = batch['label'].to(device)

        optimizer.zero_grad()
        logits = model(x, doy)
        loss = criterion(logits, y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        train_loss += loss.item() * x.size(0)

    train_loss /= len(train_loader.dataset)
    scheduler.step()

    # Evaluation
    model.eval()
    val_loss = 0.0
    all_preds, all_targets = [], []
    with torch.no_grad():
        for batch in val_loader:
            x = batch['inputs'].to(device)
            doy = batch['doy'].to(device)
            y = batch['label'].to(device)
            logits = model(x, doy)
            loss = criterion(logits, y)
            val_loss += loss.item() * x.size(0)
            preds = torch.argmax(logits, dim=1)
            all_preds.append(preds.cpu())
            all_targets.append(y.cpu())

    val_loss /= len(val_loader.dataset)
    val_preds = torch.cat(all_preds, dim=0)
    val_targets = torch.cat(all_targets, dim=0)
    metrics = compute_segmentation_metrics(val_preds, val_targets, num_classes=NUM_CLASSES, ignore_index=0)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['val_f1'].append(metrics['macro_f1'])
    history['val_iou'].append(metrics['mean_iou'])

    print(f"Epoch [{epoch:02d}/{EPOCHS}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | OA: {metrics['accuracy']:.4f} | Macro F1: {metrics['macro_f1']:.4f} | mIoU: {metrics['mean_iou']:.4f}")

    if metrics['macro_f1'] > best_f1:
        best_f1 = metrics['macro_f1']
        save_checkpoint(model, "checkpoints/best_model.pt", optimizer, epoch, metrics)

### 7. Export Best Model to ONNX

In [ ]:
# Load best checkpoint and export to ONNX for fast web deployment
checkpoint = torch.load("checkpoints/best_model.pt", map_location="cpu")
model.load_state_dict(checkpoint['model_state_dict'])

onnx_path = "checkpoints/tsvit_dinov3_crop.onnx"
export_onnx(model.cpu(), onnx_path, num_timesteps=30, img_res=24)
print(f"Model export completed: {onnx_path}")